In [ ]:
import torch
import torch.nn as nn
import timm
import types
import time
from datasets import load_dataset
from torch.utils.data import DataLoader, Subset
from torchvision import transforms
from tqdm import tqdm

# --- Configuration ---
DATA_FILES = "./data/validation-*.parquet" 
BATCH_SIZE = 64
NUM_WORKERS = 4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Ratios of heads to physically remove (0.0 = Baseline)
PRUNE_RATIOS = [0.0, 0.25, 0.50, 0.75] 

def get_dataloaders():
    print(f"Loading dataset from {DATA_FILES}...")
    dataset = load_dataset("parquet", data_files=DATA_FILES, split="train")
    
    val_transform = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    def preprocess(examples):
        images = [val_transform(img.convert('RGB')) for img in examples['image']]
        return {'image': images, 'label': examples['label']}

    dataset = dataset.with_transform(preprocess)
    
    # 500 images for Calibration (Entropy Measurement), remaining for Evaluation
    calib_indices = list(range(500))
    eval_indices = list(range(500, len(dataset)))
    
    calib_loader = DataLoader(Subset(dataset, calib_indices), batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=True)
    eval_loader = DataLoader(Subset(dataset, eval_indices), batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=True)
    
    return calib_loader, eval_loader

# --- Custom Forward Pass to Measure Entropy ---
def calibrate_attention_forward(self, x):
    """
    Temporarily replaces the forward pass of the attention block to capture entropy 
    during the 500-image calibration phase.
    """
    B, N, C = x.shape
    # Standard timm ViT qkv extraction
    qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
    q, k, v = qkv.unbind(0)
    
    q = q * self.scale
    attn = (q @ k.transpose(-2, -1))
    attn = attn.softmax(dim=-1)

    # Calculate Entropy: -sum(p * log(p))
    # A higher entropy means the head's attention is scattered/uniform (less useful)
    entropy = -(attn * torch.log(attn + 1e-8)).sum(dim=-1) 
    mean_entropy = entropy.mean(dim=(0, 2)) # Average over Batch and Sequence Length

    # Store it inside the module for later retrieval
    if not hasattr(self, 'accumulated_entropy'):
        self.accumulated_entropy = mean_entropy
        self.calib_steps = 1
    else:
        self.accumulated_entropy += mean_entropy
        self.calib_steps += 1

    attn = self.attn_drop(attn)
    x = (attn @ v).transpose(1, 2).reshape(B, N, C)
    x = self.proj(x)
    x = self.proj_drop(x)
    return x

def physically_prune_heads(model, prune_ratio):
    """
    Identifies the highest-entropy heads and physically slices their dimensions
    out of the QKV and Projection weight matrices.
    """
    for block in model.blocks:
        attn = block.attn
        num_heads = attn.num_heads
        head_dim = attn.head_dim
        
        num_keep = max(1, int(num_heads * (1 - prune_ratio)))
        if num_keep == num_heads:
            continue
            
        # 1. Rank heads by average entropy
        avg_entropy = attn.accumulated_entropy / attn.calib_steps
        
        # We want to KEEP the heads with the LOWEST entropy (most focused)
        _, sorted_indices = torch.sort(avg_entropy, descending=False)
        kept_heads = sorted_indices[:num_keep].tolist()
        kept_heads.sort() # Keep original order to preserve feature structure
        
        # 2. Build exact tensor indices for slicing
        D = num_heads * head_dim
        qkv_indices = []
        proj_indices = []
        
        for h in kept_heads:
            start = h * head_dim
            end = (h + 1) * head_dim
            
            # timm QKV shape: [Q_all, K_all, V_all]
            qkv_indices.extend(range(start, end))             # Query indices
            qkv_indices.extend(range(D + start, D + end))     # Key indices
            qkv_indices.extend(range(2*D + start, 2*D + end)) # Value indices
            
            # Projection layer takes concatenated heads as input
            proj_indices.extend(range(start, end))

        device = attn.qkv.weight.device
        qkv_idx_tensor = torch.tensor(qkv_indices, device=device)
        proj_idx_tensor = torch.tensor(proj_indices, device=device)

        # 3. Physically slice the weights (Overwriting the old tensors)
        attn.qkv.weight = nn.Parameter(torch.index_select(attn.qkv.weight, dim=0, index=qkv_idx_tensor))
        if attn.qkv.bias is not None:
            attn.qkv.bias = nn.Parameter(torch.index_select(attn.qkv.bias, dim=0, index=qkv_idx_tensor))

        attn.proj.weight = nn.Parameter(torch.index_select(attn.proj.weight, dim=1, index=proj_idx_tensor))
        # Note: proj.bias is NOT sliced because its size corresponds to the output dimension, which remains unchanged.
        
        # 4. Update the architecture definition
        attn.num_heads = num_keep
        
        # Clean up memory
        del attn.accumulated_entropy
        del attn.calib_steps

def evaluate_model(model, eval_loader, ratio_name):
    model.eval()
    correct, total = 0, 0
    timings = []
    
    starter, ender = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)

    with torch.no_grad():
        for i, batch in enumerate(tqdm(eval_loader, desc=f"Evaluating {ratio_name}")):
            inputs = batch['image'].to(DEVICE)
            targets = batch['label'].to(DEVICE)

            if i > 5: starter.record()
            outputs = model(inputs)
            if i > 5:
                ender.record()
                torch.cuda.synchronize()
                timings.append(starter.elapsed_time(ender))

            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()

    acc = 100. * correct / total
    lat = sum(timings) / len(timings) if timings else 0 
    
    # Calculate exactly how many millions of parameters are left
    params = sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6
    return acc, lat, params

def main():
    calib_loader, eval_loader = get_dataloaders()
    results = {}

    for ratio in PRUNE_RATIOS:
        print(f"\n{'='*60}")
        print(f"Testing Prune Ratio: {int(ratio*100)}% of Heads Removed")
        print(f"{'='*60}")
        
        # Load a fresh model for each sweep to avoid compounding prunes
        model = timm.create_model('vit_small_patch16_224', pretrained=True).to(DEVICE)
        
        if ratio > 0.0:
            print("[1/3] Attaching Entropy Hooks...")
            original_forwards = {}
            for i, block in enumerate(model.blocks):
                original_forwards[i] = block.attn.forward
                # Inject the custom forward pass
                block.attn.forward = types.MethodType(calibrate_attention_forward, block.attn)
                
            print("[2/3] Calibrating Entropy (500 images)...")
            with torch.no_grad():
                for batch in tqdm(calib_loader, desc="Calibrating", leave=False):
                    model(batch['image'].to(DEVICE))
                    
            print("[3/3] Slicing Weight Matrices...")
            physically_prune_heads(model, ratio)
            
            # Restore standard PyTorch forward pass for maximum inference speed
            for i, block in enumerate(model.blocks):
                block.attn.forward = original_forwards[i]
        
        acc, lat, params = evaluate_model(model, eval_loader, f"{int(ratio*100)}% Pruned")
        results[f"{int(ratio*100)}%"] = {"Accuracy": acc, "Latency (ms)": lat, "Params (M)": params}

    # --- Print Final Trade-off Matrix ---
    print(f"\n{'='*65}")
    print(f"{'Entropy Pruning Trade-off Matrix (Standard ViT)':^65}")
    print(f"{'='*65}")
    print(f"{'Pruned %':<12} | {'Accuracy (%)':<14} | {'Latency (ms)':<14} | {'Params (M)':<12}")
    print("-" * 65)
    for p, metrics in results.items():
        print(f"{p:<12} | {metrics['Accuracy']:>12.2f} % | {metrics['Latency (ms)']:>12.2f} ms | {metrics['Params (M)']:>10.2f} M")

if __name__ == "__main__":
    main()